Course: Language as Data, University of Göttingen

Last updated: October 2025

# Using mGPT for Zero-Shot and Few-Shot Translation
This notebook demonstrates how to use mGPT for Zero-Shot and Few-Shot translation tasks.
We use the `ai-forever/mGPT` model from Hugging Face's Transformers library.

## Installs & model

In [ ]:
# Install nltk library if not already installed
!pip install nltk

In [ ]:
# Install required libraries
# !pip install "ipykernel==6.30.1"
# !pip install transformers
# !pip install --upgrade accelerate
# !pip install sentencepiece
# !pip install -U huggingface_hub

Note: loading this model of 7 GB make take several minutes, depending on your download speed.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Load mGPT model and tokenizer
#model_name = "openGPT-X/Teuken-7B-instruct-v0.6"
model_name = 'ai-forever/mGPT'
#model_name = 'bigscience/bloomz-560m'
#model_name = 'facebook/xglm-564M'
tokenizer = AutoTokenizer.from_pretrained(model_name) # , use_fast=True, progress_bar=False
print('Loaded tokenizer.')
model = AutoModelForCausalLM.from_pretrained(model_name) # , use_fast=True, progress_bar=False
print('Loaded model.')

# Set device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device: {device}")
model.to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

Loaded tokenizer.


config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

Loaded model.
device: cuda


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(100000, 2048)
    (wpe): Embedding(2048, 2048)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=6144, nx=2048)
          (c_proj): Conv1D(nf=2048, nx=2048)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=8192, nx=2048)
          (c_proj): Conv1D(nf=2048, nx=8192)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=2048, out_features=100000, bias=False)
)

In [ ]:
print(model.config)


GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 5,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 2048,
  "n_embd": 2048,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 24,
  "n_positions": 2048,
  "pad_token_id": 1,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 100000
}



## Zero-Shot Translation
We translate an English sentence to German without providing any examples.

In [ ]:
# Define the Zero-Shot prompt
sentence_en = "My name is Tolga."
zero_shot_prompt = f"English: {sentence_en}\nFrançais:"

# Tokenize input
inputs = tokenizer(zero_shot_prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        num_beams=5,
        eos_token_id=tokenizer.eos_token_id
    )

gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# post-process output
if "Français:" in gen_text:
  gen_text = gen_text.split("Français:")[1].split("\n")[0]

print("Final Zero-Shot Translation:", gen_text.strip())

Final Zero-Shot Translation: Mon nom est Tolga.


## Europarl dataset

In [ ]:
!wget https://www.statmt.org/europarl/v10/training/europarl-v10.fr-en.tsv.gz
!gunzip europarl-v10.fr-en.tsv.gz

--2026-01-20 23:02:17--  https://www.statmt.org/europarl/v10/training/europarl-v10.fr-en.tsv.gz
Resolving www.statmt.org (www.statmt.org)... 129.215.32.28
Connecting to www.statmt.org (www.statmt.org)|129.215.32.28|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218267083 (208M) [application/x-gzip]
Saving to: ‘europarl-v10.fr-en.tsv.gz’

europarl-v10.fr-en. 100%[===================>] 208.16M  23.6MB/s    in 9.9s    

2026-01-20 23:02:27 (21.1 MB/s) - ‘europarl-v10.fr-en.tsv.gz’ saved [218267083/218267083]



parse europarl-v10.fr-en.tsv

In [ ]:
import pandas as pd

# Define column names based on the readme
column_names = ['source', 'target', 'file_id', 'chapter_id', 'speaker_id', 'speaker_name', 'language', 'affiliation']

# Read the TSV file, specifying the separator as tab, with explicit column names and error handling.
# header=None indicates that there is no header row in the file.
# on_bad_lines='skip' will skip any lines that have an incorrect number of fields.
df = pd.read_csv('europarl-v10.fr-en.tsv', sep='\t', names=column_names, header=None, on_bad_lines='skip')

# Display the first 5 rows of the 'source' (French) and 'target' (English) columns
print("First 5 lines of 'source' (French) and 'target' (English) text:")
display(df[['source', 'target']].head())

/tmp/ipython-input-4037980013.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('europarl-v10.fr-en.tsv', sep='\t', names=column_names, header=None, on_bad_lines='skip')


First 5 lines of 'source' (French) and 'target' (English) text:


,source,target
0,4.,4.
1,Ratification et mise en œuvre des conventions ...,The ratification and implementation of the upd...
2,7.,7.
3,Agriculture durable et biogaz: nécessité de re...,Sustainable agriculture and biogas: review of ...
4,- Avant le vote sur le paragraphe 41:,- Before the vote on paragraph 41:


filter out the rows where either the 'source' or 'target' column contains only numbers.

In [ ]:
import re

# Function to check if a string is purely numeric (including decimals)
def is_numeric_only(text):
    if isinstance(text, str):
        return re.fullmatch(r'\d+\.?\d*', text.strip()) is not None
    return False

# Filter out rows where 'source' or 'target' are purely numeric
df_filtered = df[~df['source'].apply(is_numeric_only) & ~df['target'].apply(is_numeric_only)]

print("First 5 lines of the DataFrame after filtering numeric-only rows:")
display(df_filtered[['source', 'target']].head())

First 5 lines of the DataFrame after filtering numeric-only rows:


,source,target
1,Ratification et mise en œuvre des conventions ...,The ratification and implementation of the upd...
3,Agriculture durable et biogaz: nécessité de re...,Sustainable agriculture and biogas: review of ...
4,- Avant le vote sur le paragraphe 41:,- Before the vote on paragraph 41:
5,"rapporteur. - (EN) Monsieur le Président, nous...","rapporteur. - Mr President, we agreed to chang..."
6,"Le texte est le suivant: ""Propose l'insertion ...",The text is as follows: 'Proposes the comprehe...


## First few rows

In [ ]:
translation_results = []

# Iterate over the first 5 rows of the filtered DataFrame for demonstration
for index, row in df_filtered.head(5).iterrows():
    original_french = row['source']
    original_english = row['target']

    # Prepare the zero-shot prompt for translation from English to French
    zero_shot_prompt = f"English: {original_english}\nFrançais:"

    # Tokenize input
    inputs = tokenizer(zero_shot_prompt, return_tensors="pt").to(device)

    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            num_beams=5,
            eos_token_id=tokenizer.eos_token_id
        )

    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Post-process output to extract the French translation
    generated_french = ""
    if "Français:" in gen_text:
        # Split by "Français:" and take the second part, then split by newline to get the first line of translation
        generated_french = gen_text.split("Français:")[1].split("\n")[0].strip()

    # Store results
    translation_results.append({
        "original_english": original_english,
        "original_french": original_french,
        "generated_french": generated_french
    })

# Print the accumulated translation results
print("Translation Results (English, Original French, Generated French):")
for result in translation_results:
    print(f"\nOriginal English: {result['original_english']}")
    print(f"Original French (from dataset): {result['original_french']}")
    print(f"Generated French: {result['generated_french']}")


Translation Results (English, Original French, Generated French):

Original English: The ratification and implementation of the updated ILO conventions (vote)
Original French (from dataset): Ratification et mise en œuvre des conventions de l'OIT mises à jour (vote)
Generated French: La ratification et l'application des conventions de l'Organisation internationale du travail (vote)

Original English: Sustainable agriculture and biogas: review of EU legislation (
Original French (from dataset): Agriculture durable et biogaz: nécessité de revoir la législation communautaire (
Generated French: Agriculture biologique et biogaz: revue de la législation européenne (

Original English: - Before the vote on paragraph 41:
Original French (from dataset): - Avant le vote sur le paragraphe 41:
Generated French: - Avant le vote sur le paragraphe 41:

Original English: rapporteur. - Mr President, we agreed to change paragraph 41 and not to propose a specific biogas EU directive but to integrate it i

## Run inference & calculate BLEU

In [ ]:
import nltk
# Download the 'punkt' tokenizer data, required for word_tokenize
nltk.download('punkt')
# Download 'punkt_tab' which is also required by word_tokenize for some languages
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import tqdm

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import SmoothingFunction
import numpy as np

MAX_NEW_TOKENS = 70

bleu_scores = []

# Limit to the first 500 entries for BLEU score calculation
num_entries_for_bleu = 500

likely_truncated_count = 0
early_stop_count = 0

original_english_sentences = []
original_french_sentences = []
generated_french_sentences = []
likely_truncated_mask = []

length_ratios = []

# Iterate over the specified number of rows of the filtered DataFrame
print(f"Calculating BLEU scores for the first {num_entries_for_bleu} entries...")
for index, row in tqdm.tqdm(df_filtered.head(num_entries_for_bleu).iterrows(), total=num_entries_for_bleu):
    original_french = row['source']
    original_english = row['target']

    original_english_sentences.append(original_english)
    original_french_sentences.append(original_french)

    # Prepare the zero-shot prompt for translation from English to French
    zero_shot_prompt = f"English: {original_english}\nFrançais:"

    # Tokenize input
    inputs = tokenizer(zero_shot_prompt, return_tensors="pt").to(device)

    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=5,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Post-process output to extract the French translation
    generated_french = ""
    if "Français:" in gen_text:
        generated_french = gen_text.split("Français:")[1].split("\n")[0].strip()
    else:
        # Fallback if 'Français:' is not found in the generated text
        generated_french = gen_text.strip()

    generated_french_sentences.append(generated_french)

    gen_ids = tokenizer(
        generated_french,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"][0]

    num_gen_tokens = gen_ids.shape[0]

    likely_truncated = num_gen_tokens >= MAX_NEW_TOKENS

    if likely_truncated:
        likely_truncated_count += 1
        likely_truncated_mask.append(True)
    else:
        likely_truncated_mask.append(False)
        early_stop_count += 1

    # Tokenize the reference and candidate sentences for BLEU score calculation
    # The reference is a list of tokenized sentences, as sentence_bleu expects multiple references
    reference = [word_tokenize(original_french.lower())]
    candidate = word_tokenize(generated_french.lower())

    # calculate length ratios
    # tokenized already for BLEU
    ref_len = len(reference[0])        # original_french
    gen_len = len(candidate)           # generated_french

    length_ratio = gen_len / ref_len if ref_len > 0 else 0.0
    length_ratios.append(length_ratio)


    # Calculate BLEU score. Using weights=(1.0, 0.0, 0.0, 0.0) is for unigram BLEU, which is less sensitive.
    # For a more standard BLEU, use default weights=(0.25, 0.25, 0.25, 0.25).
    # Using default weights here.
    score = sentence_bleu(reference, candidate,
                          smoothing_function=SmoothingFunction().method1)
    bleu_scores.append(score)



Calculating BLEU scores for the first 500 entries...


100%|██████████| 500/500 [21:05<00:00,  2.53s/it]


In [ ]:
print(f"\nStopped with EOS count: {early_stop_count}")
print(f"Hit max_new_tokens count: {likely_truncated_count}")
# Calculate the average BLEU score
if not bleu_scores:
  print("No BLEU scores were calculated.")
else:
  print(f"Metrics over {num_entries_for_bleu} outputs:")
  average_bleu = np.mean(bleu_scores)
  print(f"\nAverage BLEU score : {average_bleu:.4f}")
  median_bleu = np.median(bleu_scores)
  print(f"Median BLEU score: {median_bleu:.4f}")

  print("--"*20)

  print(f"Metrics excluding likely truncated outputs:")
  average_bleu = np.mean(np.array(bleu_scores)[~np.array(likely_truncated_mask)])
  print(f"\nAverage BLEU score : {average_bleu:.4f}")
  median_bleu = np.median(np.array(bleu_scores)[~np.array(likely_truncated_mask)])
  print(f"Median BLEU score: {median_bleu:.4f}")

  print("--"*20)

  print(f"Average length ratio: {np.mean(length_ratios):.4f}")
  print(f"Median length ratio: {np.median(length_ratios):.4f}")

  print("--"*20)
  zero_bleu_count = np.sum(np.array(bleu_scores) == 0)
  print(f"zero BLEU count: {zero_bleu_count}")
  print(f"zero BLEU %: {zero_bleu_count / len(bleu_scores):.4f}")

  print("--"*20)

  print(f"5 best scoring translations:")
  sorted_indices = np.argsort(bleu_scores)[::-1]
  for i in range(5):
    idx = sorted_indices[i]
    print(f"Original English: {original_english_sentences[idx]}")
    print(f"Original French (from dataset): {original_french_sentences[idx]}")
    print(f"Generated French: {generated_french_sentences[idx]}")
    print(f"BLEU score: {bleu_scores[idx]:.4f}")
    print()

  print("--"*20)

  print(f"5 worst scoring translations:")
  sorted_indices = np.argsort(bleu_scores)
  for i in range(5):
    idx = sorted_indices[i]
    print(f"Original English: {original_english_sentences[idx]}")
    print(f"Original French (from dataset): {original_french_sentences[idx]}")
    print(f"Generated French: {generated_french_sentences[idx]}")
    print(f"BLEU score: {bleu_scores[idx]:.4f}")
    print()


Stopped with EOS count: 452
Hit max_new_tokens count: 48
Metrics over 500 outputs:

Average BLEU score : 0.2614
Median BLEU score: 0.2353
----------------------------------------
Metrics excluding likely truncated outputs:

Average BLEU score : 0.2652
Median BLEU score: 0.2402
----------------------------------------
Average length ratio: 0.9203
Median length ratio: 0.9304
----------------------------------------
zero BLEU count: 1
zero BLEU %: 0.0020
----------------------------------------
5 best scoring translations:
Original English: - Before the vote on paragraph 41:
Original French (from dataset): - Avant le vote sur le paragraphe 41:
Generated French: - Avant le vote sur le paragraphe 41:
BLEU score: 1.0000

Original English: The debate is closed.
Original French (from dataset): Le débat est clos.
Generated French: Le débat est clos.
BLEU score: 1.0000

Original English: That is very clear.
Original French (from dataset): C'est très clair.
Generated French: C'est très clair.
BL